# Qa runtime compatibility

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA Runtime Compatibility

Deterministic smoke checks for model/runtime/loss/checkpoint compatibility.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import yaml

from src.data_pipeline import load_point_centric_arrays, build_split_dataloader
from src.models import build_model_from_config
from src.losses import build_loss_from_config
from src.train import build_optimizer

ROOT = Path.cwd()
if (
    not (ROOT / "configs" / "training.yaml").exists()
    and (ROOT.parent / "configs" / "training.yaml").exists()
):
    ROOT = ROOT.parent

cfg = yaml.safe_load((ROOT / "configs" / "training.yaml").read_text()) or {}
data_cfg = cfg.get("data", {}) or {}
log_cfg = cfg.get("logging", {}) or {}

checks = []


def record(name, ok, detail=""):
    checks.append({"check": str(name), "ok": bool(ok), "detail": str(detail)})


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
record("device_selected", True, str(device))

In [ ]:
pc_dir = ROOT / str(data_cfg.get("point_centric_dir", "data/processed/point_centric_demo"))
record("point_centric_dir_exists", pc_dir.exists(), str(pc_dir))

if pc_dir.exists():
    arrays = load_point_centric_arrays(str(pc_dir))
    train_loader, train_ds = build_split_dataloader(arrays, cfg, split_name="train", shuffle=False)
    record("train_dataset_non_empty", len(train_ds) > 0, f"len={len(train_ds)}")

    if len(train_ds) > 0:
        sample = train_ds[0]
        model = build_model_from_config(
            config=cfg,
            dynamic_input_dim=int(sample["x_dynamic"].shape[-1]),
            static_input_dim=int(sample["x_static"].shape[-1]),
            output_dim=int(sample["y"].shape[-1]),
        ).to(device)

        x_dyn = sample["x_dynamic"].unsqueeze(0).to(device)
        x_stat = sample["x_static"].unsqueeze(0).to(device)
        y_true = sample["y"].unsqueeze(0).to(device)

        y_pred = model(x_dyn, x_stat)
        record(
            "forward_shape_contract", tuple(y_pred.shape) == (1, 6), f"shape={tuple(y_pred.shape)}"
        )

        loss_fn = build_loss_from_config(cfg)
        loss_params = [p for p in loss_fn.parameters() if p.requires_grad]
        optimizer, opt_kind = build_optimizer(model, cfg, extra_parameters=loss_params)

        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(y_pred, y_true)
        loss.backward()
        optimizer.step()

        record(
            "loss_backward_step",
            bool(torch.isfinite(loss.detach()).item()),
            f"loss={float(loss.detach()):.6f}, optimizer={opt_kind}",
        )
        if loss_params:
            grads_ok = all(p.grad is not None for p in loss_params)
            record(
                "trainable_loss_params_grad",
                grads_ok,
                f"count={sum(p.numel() for p in loss_params)}",
            )

        ckpt_path = (
            ROOT
            / str(log_cfg.get("output_dir", "results/dual_branch"))
            / str(log_cfg.get("checkpoint_name", "dual_branch_best.pt"))
        )
        if not ckpt_path.exists():
            record("checkpoint_load_compatibility", True, f"SKIP: {ckpt_path} not found")
        else:
            checkpoint = torch.load(ckpt_path, map_location="cpu")
            if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
                state = checkpoint["model_state_dict"]
            elif (
                isinstance(checkpoint, dict)
                and checkpoint
                and all(torch.is_tensor(v) for v in checkpoint.values())
            ):
                state = checkpoint
            else:
                state = None

            if state is None:
                record(
                    "checkpoint_format_supported", False, f"unsupported format: {type(checkpoint)}"
                )
            else:
                try:
                    model.load_state_dict(state)
                    record("checkpoint_strict_load", True, f"path={ckpt_path}")
                except RuntimeError:
                    current = model.state_dict()
                    compatible = {
                        k: v
                        for k, v in state.items()
                        if k in current and current[k].shape == v.shape
                    }
                    if compatible:
                        model.load_state_dict(compatible, strict=False)
                        record("checkpoint_partial_load", True, f"loaded={len(compatible)}")
                    else:
                        record("checkpoint_partial_load", False, "no shape-compatible tensors")

summary = pd.DataFrame(checks)
print(summary.to_string(index=False))
failed = summary[summary["ok"] == False]
assert failed.empty, f"{len(failed)} runtime compatibility checks failed"
print("Runtime compatibility QA passed.")